In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 2: Data Cleaning -> olist_order_items_dataset.csv (for Tableau dashboard)
==================================================================================
 Purpose : Clean the order line-item fact table for use in Tableau, joined to
           orders (order_id), products (product_id), and sellers (seller_id).

 IMPORTANT NOTES FOR A BI/DASHBOARD DATASET:
   1. This is the FACT table (transactional grain: one row per order line
      item). Rows must never be dropped for "outlier" prices/freight, since
      a genuinely expensive order is real revenue, not an error -> removing
      it would understate revenue KPIs on the dashboard. Outliers are only
      flagged/reported here, never deleted.
   2. 'shipping_limit_date' arrives as text -> parsed into a proper datetime
      so Tableau recognizes it as a date field (enabling date hierarchies,
      trend lines, etc.) instead of treating it as a plain string.
   3. This file had no missing values or duplicate line items -> cleaning
      here is mostly type correction and validation, not repair.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_order_items_dataset.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/olist_order_items_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("2. DUPLICATE RECORD CHECK")

full_dupes = df.duplicated().sum()
line_item_dupes = df.duplicated(subset=["order_id", "order_item_id"]).sum()

print(f"Fully duplicated rows                        : {full_dupes}")
print(f"Duplicated (order_id + order_item_id) combos : {line_item_dupes}")

before_rows = df.shape[0]
df = df.drop_duplicates()   # safety net; confirmed 0 found
after_rows = df.shape[0]
print(f"Rows removed as duplicates                    : {before_rows - after_rows}")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE CHECK
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE CHECK")

missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values found. No imputation needed for this file.")
else:
    print(missing)


# ----------------------------------------------------------------------------------
# 4. DATA TYPE CORRECTION (date parsing)
# ----------------------------------------------------------------------------------
section("4. DATA TYPE CORRECTION")

print(f"'shipping_limit_date' dtype before parsing: {df['shipping_limit_date'].dtype}")
df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"])
print(f"'shipping_limit_date' dtype after parsing : {df['shipping_limit_date'].dtype}")


# ----------------------------------------------------------------------------------
# 5. INVALID VALUE CHECK (price / freight must be non-negative)
# ----------------------------------------------------------------------------------
section("5. INVALID VALUE CHECK")

invalid_price = (df["price"] <= 0).sum()
invalid_freight = (df["freight_value"] < 0).sum()
print(f"Rows with price <= 0        : {invalid_price}")
print(f"Rows with freight_value < 0 : {invalid_freight}")
print("(None found -> no rows require correction here.)")


# ----------------------------------------------------------------------------------
# 6. OUTLIER FLAGGING (reported only, not removed - see notes above)
# ----------------------------------------------------------------------------------
section("6. OUTLIER FLAGGING (FOR AWARENESS, NOT REMOVAL)")

for col in ["price", "freight_value"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    outlier_count = (df[col] > upper_bound).sum()
    print(f"{col}: IQR upper bound = {round(upper_bound, 2)} -> "
          f"{outlier_count} rows above it ({round(outlier_count / len(df) * 100, 2)}%). "
          f"Kept in the dataset -> these are legitimate high-value transactions.")


# ----------------------------------------------------------------------------------
# 7. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("7. FINAL VALIDATION")

print(f"Final shape                       : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining missing values          : {df.isnull().sum().sum()}")
print(f"Remaining duplicate rows          : {df.duplicated().sum()}")
print(f"Row count unchanged from raw file : {df.shape[0] == before_rows}")
print(f"'shipping_limit_date' is datetime : {pd.api.types.is_datetime64_any_dtype(df['shipping_limit_date'])}")


# ----------------------------------------------------------------------------------
# 8. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("8. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("ORDER ITEMS DATASET CLEANING COMPLETE")
print("This file is now safe to load into Tableau as the central fact table,")
print("joined to orders, products, and sellers on their respective ID columns.")


 1. LOAD RAW DATA
Raw dataset loaded: 112650 rows, 7 columns

 2. DUPLICATE RECORD CHECK
Fully duplicated rows                        : 0
Duplicated (order_id + order_item_id) combos : 0
Rows removed as duplicates                    : 0

 3. MISSING VALUE CHECK
No missing values found. No imputation needed for this file.

 4. DATA TYPE CORRECTION
'shipping_limit_date' dtype before parsing: str
'shipping_limit_date' dtype after parsing : datetime64[us]

 5. INVALID VALUE CHECK
Rows with price <= 0        : 0
Rows with freight_value < 0 : 0
(None found -> no rows require correction here.)

 6. OUTLIER FLAGGING (FOR AWARENESS, NOT REMOVAL)
price: IQR upper bound = 277.4 -> 8427 rows above it (7.48%). Kept in the dataset -> these are legitimate high-value transactions.
freight_value: IQR upper bound = 33.25 -> 11613 rows above it (10.31%). Kept in the dataset -> these are legitimate high-value transactions.

 7. FINAL VALIDATION
Final shape                       : 112650 rows, 7 columns
R